In [ ]:
def parse_hip_csv(filepath):
    """Parse the entire CSV into a dataframe."""
    df = pd.read_csv(filepath)
    return df

def prepare_plot_data(df):
    """Extract and restructure data for plotting."""

    data = {
        'Configuration': [],
        'Target Firing Rate (%)': [],
        'Accuracy (%)': [],
        'Precision (%)': [],
        'Recall (%)': [],
        'F1-Score (%)': [],
        'Hidden Sparsity (%)': [],
        'Output Sparsity (%)': [],
        'Dead Neurons (hidden)': [],
        'Saturated Neurons (hidden)': []
    }

    current_config = None

    for idx, row in df.iterrows():
        first_col = str(row.iloc[0]).strip() if pd.notna(row.iloc[0]) else ""

        # Check if this is a configuration header
        if '=' in first_col and not first_col.startswith('seed') and not first_col.startswith('Target'):
            current_config = first_col.strip('"')
            continue
        
        # Assign configuration for baseline 'seed' rows
        if first_col.startswith('seed'):
            current_config = 'Hidden = Leaky, Out = Leaky'

        # Skip header rows
        if 'Accuracy' in first_col or 'Target Firing Rate' in first_col:
            continue

        # Skip empty rows
        if pd.isna(row.iloc[0]) or first_col == '':
            continue

        # Determine if this is a seed or target firing rate
        target_rate = None
        if first_col.startswith('seed'):
            target_rate = 0  # Baseline seeds assigned to 0%
        elif '%' in first_col:
            try:
                target_rate = int(first_col.rstrip('%'))
            except ValueError:
                continue
        else:
            continue

        if target_rate is None: # Skip if target_rate couldn't be determined
            continue

        # Extract all metrics
        try:
            row_data = {
                'Configuration': current_config,
                'Target Firing Rate (%)': target_rate,
                'Accuracy (%)': float(str(row.iloc[1]).rstrip('%')),
                'Precision (%)': float(str(row.iloc[2]).rstrip('%')),
                'Recall (%)': float(str(row.iloc[3]).rstrip('%')),
                'F1-Score (%)': float(str(row.iloc[4]).rstrip('%')),
                'Hidden Sparsity (%)': float(str(row.iloc[5]).rstrip('%')),
                'Output Sparsity (%)': float(str(row.iloc[6]).rstrip('%')),
                'Dead Neurons (hidden)': int(row.iloc[7]),
                'Saturated Neurons (hidden)': int(row.iloc[8])
            }
            
            # Ensure the config is not None for data points
            if row_data['Configuration'] is None:
                continue

            for key, value in row_data.items():
                data[key].append(value)
        except (ValueError, TypeError):
            continue

    plot_df = pd.DataFrame(data)
    return plot_df

def plot_metric(plot_df, metric='Accuracy (%)', output_path=None):
    """
    Plot any metric against target firing rate with 95% CI.
    """

    fig, ax = plt.subplots(figsize=(11, 6))

    # Define colors with explicit z-order control
    palette = {
        'Hidden = Leaky, Out = Leaky': 'gray',
        'Hidden = LIF-HIP, Out = Leaky': 'steelblue',
        'Hidden = LIF-HIP, Out = LIF-HIP': 'coral'
    }

    # Separate baseline data from HIP data
    baseline_config_name = 'Hidden = Leaky, Out = Leaky'
    baseline_data = plot_df[(plot_df['Configuration'] == baseline_config_name) &
                            (plot_df['Target Firing Rate (%)'] == 0)]
    
    # Calculate baseline mean and CI
    baseline_mean = baseline_data[metric].mean()
    baseline_ci = baseline_data[metric].std() * 1.96 / np.sqrt(len(baseline_data)) if len(baseline_data) > 1 else 0

    # Plot baseline as horizontal line with shaded CI region
    ax.axhline(y=baseline_mean, color='gray', linestyle='--', linewidth=2.5,
               label=baseline_config_name, zorder=1)

    # Filter out baseline for lineplot
    plot_df_hip = plot_df[plot_df['Configuration'] != baseline_config_name].copy()

    # Ensure only the HIP configurations are in the hue_order for sns.lineplot
    hip_configs = [c for c in palette if c != baseline_config_name]
    ordered_hip_configs = [config for config in hip_configs if config in plot_df_hip['Configuration'].unique()]
    
    # Plot with seaborn for HIP configurations
    if not plot_df_hip.empty:
        sns.lineplot(data=plot_df_hip, x='Target Firing Rate (%)', y=metric,
                     hue='Configuration', style='Configuration',
                     markers=True, markersize=9, linewidth=2.5, dashes=False,
                     errorbar=('ci', 95),
                     palette={k: v for k, v in palette.items() if k != baseline_config_name},
                     hue_order=ordered_hip_configs,
                     ax=ax)
        ax.set_xlim([-0.5, 15.5])

    # add the baseline CI shading after plotting
    if len(baseline_data) > 1:
        xlim = ax.get_xlim()
        ax.fill_between([xlim[0], xlim[1]], baseline_mean - baseline_ci, baseline_mean + baseline_ci,
                         alpha=0.15, color='gray', zorder=0)

    ax.set_title(f'SNN {metric} vs. Target Firing Rate',
                 fontsize=14, fontweight='bold')
    ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    ax.set_xlabel('Target Firing Rate (%)', fontsize=12, fontweight='bold')
    ax.set_xticks([0, 5, 10, 15])
    # Dynamically set y-limits based on the metric and data, but keep it reasonable
    min_val = plot_df[metric].min() * 0.95 if plot_df[metric].min() > 0 else plot_df[metric].min() * 1.05
    max_val = plot_df[metric].max() * 1.05
    
    #set 100% as maximum for all % based values
    if '%' in metric:
      max_val = min(max_val, 100)
    ax.set_ylim([min_val, max_val])

    ax.grid(True, alpha=0.3, linestyle=':')
    ax.legend(loc='best', fontsize=11, framealpha=0.95)

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"✓ Plot saved to {output_path}")
    plt.show()

In [ ]:
# Usage
if __name__ == '__main__':
    # Step 1: Parse entire CSV
    df_raw = parse_hip_csv('/content/SNN_data.csv')

    # Step 2: Prepare data for plotting
    plot_df = prepare_plot_data(df_raw)
    print("Prepared data:")
    print(plot_df.groupby('Configuration')['Target Firing Rate (%)'].unique())

    # Step 3: Plot different metrics
    plot_metric(plot_df, metric='Accuracy (%)', output_path='accuracy_comparison.png')
    plot_metric(plot_df, metric='F1-Score (%)', output_path='f1_comparison.png')
    plot_metric(plot_df, metric='Hidden Sparsity (%)', output_path='sparsity_comparison.png')
    plot_metric(plot_df, metric='Output Sparsity (%)', output_path='sparsity_comparison.png')
    plot_metric(plot_df, metric='Dead Neurons (hidden)', output_path='dead_neurons_comparison.png')